# Prompt Engineering Best Practices for Instruction-Tuned LLM [Part 1]

[Article](https://todatabeyond.com/blog/prompt-engineering-best-practices-for-instruction-tuned-llm-part-1). [Series contents](./README.md).

Install `pip install openai==2.43.0 python-dotenv==1.2.2`. Set OPENAI_API_KEY in your environment (or an uncommitted .env); API calls incur charges. MODEL defaults to the tested gpt-5.4-mini. Never place credentials in cells. The examples request ordinary task outputs; responses can change between runs. The nonexecuting appendix preserves original source code and transcripts.

In [ ]:
import os, json, re, sys
from importlib.metadata import version
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()
MODEL = os.getenv("MODEL", "gpt-5.4-mini")
client = OpenAI(max_retries=0, timeout=60)
def get_completion(prompt, model=MODEL):
    result = client.chat.completions.create(
        model=model, messages=[{"role": "user", "content": prompt}],
        reasoning_effort="none", max_completion_tokens=2048)
    if result.choices[0].finish_reason != "stop":
        raise RuntimeError("The completion did not finish normally")
    content = result.choices[0].message.content
    if not content:
        raise RuntimeError("No text returned")
    return content
def parse_json_object(text):
    decoder = json.JSONDecoder()
    for index, char in enumerate(text):
        if char == "{":
            try:
                value, _ = decoder.raw_decode(text[index:])
            except json.JSONDecodeError:
                continue
            if isinstance(value, dict):
                return value
    raise ValueError("No JSON object found")
print("Python:", sys.version.split()[0])
print("openai:", version("openai"), "python-dotenv:", version("python-dotenv"))
print("Model:", MODEL)

In [ ]:
text = f"""
You should express what you want a model to do by  providing instructions that are as clear and  specific as you can possibly make them.  This will guide the model towards the desired output,  and reduce the chances of receiving irrelevant  or incorrect responses. Don't confuse writing a  clear prompt with writing a short prompt.  In many cases, longer prompts provide more clarity  and context for the model, which can lead to  more detailed and relevant outputs.
"""
prompt = f"""
Summarize the text delimited by triple backticks  into a single sentence.
```{text}```
"""
response = get_completion(prompt)
print(response)
delimiter_response = response

In [ ]:
prompt = f"""
Generate a list of three made-up book titles along  with their authors and genres. 
Provide them in JSON format with the following keys: 
book_id, title, author, genre.
"""
prompt += '\nReturn only a JSON object with a books array. Each book must contain book_id (integer), title, author and genre.'
response = get_completion(prompt)
print(response)
books = parse_json_object(response)

In [ ]:
text_1 = f"""
Making a cup of tea is easy! First, you need to get some  water boiling. While that's happening,  grab a cup and put a tea bag in it. Once the water is  hot enough, just pour it over the tea bag.  Let it sit for a bit so the tea can steep. After a  few minutes, take out the tea bag. If you  like, you can add some sugar or milk to taste.  And that's it! You've got yourself a delicious  cup of tea to enjoy.
"""
prompt = f"""
You will be provided with text delimited by triple quotes. 
If it contains a sequence of instructions,  re-write those instructions in the following format:

Step 1 - ...
Step 2 - …
…
Step N - …

If the text does not contain a sequence of instructions,  then simply write \"No steps provided.\"

\"\"\"{text_1}\"\"\"
"""
response = get_completion(prompt)
print("Completion for Text 1:")
print(response)
tea_response = response

In [ ]:
text_2 = f"""
The sun is shining brightly today, and the birds are  singing. It's a beautiful day to go for a  walk in the park. The flowers are blooming, and the  trees are swaying gently in the breeze. People  are out and about, enjoying the lovely weather.  Some are having picnics, while others are playing  games or simply relaxing on the grass. It's a  perfect day to spend time outdoors and appreciate the  beauty of nature.
"""
prompt = f"""
You will be provided with text delimited by triple quotes. 
If it contains a sequence of instructions,  re-write those instructions in the following format:

Step 1 - ...
Step 2 - …
…
Step N - …

If the text does not contain a sequence of instructions,  then simply write \"No steps provided.\"

\"\"\"{text_2}\"\"\"
"""
response = get_completion(prompt)
print("Completion for Text 2:")
print(response)
no_steps_response = response

In [ ]:
prompt = f"""
Your task is to answer in a consistent style.

<child>: Teach me about patience.

<grandparent>: The river that carves the deepest  valley flows from a modest spring; the  grandest symphony originates from a single note;  the most intricate tapestry begins with a solitary thread.

<child>: Teach me about resilience.
"""
response = get_completion(prompt)
print(response)
few_shot_response = response

In [ ]:
assert "clear" in delimiter_response.lower() or "specific" in delimiter_response.lower()
assert len(books["books"]) == 3
assert all(set(b) == {"book_id", "title", "author", "genre"} for b in books["books"])
assert len({b["book_id"] for b in books["books"]}) == 3
assert all(isinstance(b["book_id"], int) and all(isinstance(b[k], str) and b[k] for k in ["title", "author", "genre"]) for b in books["books"])
assert "Step 1" in tea_response and "tea" in tea_response.lower()
assert no_steps_response.lower().strip().rstrip(".") == "no steps provided"
assert len(few_shot_response.split()) > 15
assert parse_json_object('prefix {"ok": true} suffix') == {"ok": True}
try: parse_json_object("not JSON")
except ValueError: pass
else: raise AssertionError("Invalid JSON accepted")
print("PASS: summary content, all three book records, tea instructions, no-instructions branch, few-shot response and JSON parser boundary.")

## Original source appendix — not executed

Historical SDK calls and transcripts below are retained verbatim for comparison, not used as fresh test results.

### Original pre block 1

```text
pip install openai
```

### Original pre block 2

```text
import openai
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

openai.api_key  = os.getenv('OPENAI_API_KEY')
```

### Original pre block 3

```text
def get_completion(prompt, model="gpt-3.5-turbo"):
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0
    )
    return response.choices[0].message.content
```

### Original pre block 4

```text
text = f"""
You should express what you want a model to do by \ 
providing instructions that are as clear and \ 
specific as you can possibly make them. \ 
This will guide the model towards the desired output, \ 
and reduce the chances of receiving irrelevant \ 
or incorrect responses. Don't confuse writing a \ 
clear prompt with writing a short prompt. \ 
In many cases, longer prompts provide more clarity \ 
and context for the model, which can lead to \ 
more detailed and relevant outputs.
"""
prompt = f"""
Summarize the text delimited by triple backticks \ 
into a single sentence.
```{text}```
"""
response = get_completion(prompt)
print(response)
```

### Original pre block 5

```text
prompt = f"""
Generate a list of three made-up book titles along \ 
with their authors and genres. 
Provide them in JSON format with the following keys: 
book_id, title, author, genre.
"""
response = get_completion(prompt)
print(response)
```

### Original pre block 6

```text
{
  "books": [
    {
      "book_id": 1,
      "title": "The Enigma of Elysium",
      "author": "Evelyn Sinclair",
      "genre": "Mystery"
    },
    {
      "book_id": 2,
      "title": "Whispers in the Wind",
      "author": "Nathaniel Blackwood",
      "genre": "Fantasy"
    },
    {
      "book_id": 3,
      "title": "Echoes of the Past",
      "author": "Amelia Hart",
      "genre": "Romance"
    }
  ]
}
```

### Original pre block 7

```text
text_1 = f"""
Making a cup of tea is easy! First, you need to get some \ 
water boiling. While that's happening, \ 
grab a cup and put a tea bag in it. Once the water is \ 
hot enough, just pour it over the tea bag. \ 
Let it sit for a bit so the tea can steep. After a \ 
few minutes, take out the tea bag. If you \ 
like, you can add some sugar or milk to taste. \ 
And that's it! You've got yourself a delicious \ 
cup of tea to enjoy.
"""
prompt = f"""
You will be provided with text delimited by triple quotes. 
If it contains a sequence of instructions, \ 
re-write those instructions in the following format:

Step 1 - ...
Step 2 - …
…
Step N - …

If the text does not contain a sequence of instructions, \ 
then simply write \"No steps provided.\"

\"\"\"{text_1}\"\"\"
"""
response = get_completion(prompt)
print("Completion for Text 1:")
print(response)
```

### Original pre block 8

```text
Completion for Text 1:
Step 1 - Get some water boiling.
Step 2 - Grab a cup and put a tea bag in it.
Step 3 - Once the water is hot enough, pour it over the tea bag.
Step 4 - Let it sit for a bit so the tea can steep.
Step 5 - After a few minutes, take out the tea bag.
Step 6 - If you like, add some sugar or milk to taste.
Step 7 - Enjoy your delicious cup of tea.
```

### Original pre block 9

```text
text_2 = f"""
The sun is shining brightly today, and the birds are \
singing. It's a beautiful day to go for a \ 
walk in the park. The flowers are blooming, and the \ 
trees are swaying gently in the breeze. People \ 
are out and about, enjoying the lovely weather. \ 
Some are having picnics, while others are playing \ 
games or simply relaxing on the grass. It's a \ 
perfect day to spend time outdoors and appreciate the \ 
beauty of nature.
"""
prompt = f"""
You will be provided with text delimited by triple quotes. 
If it contains a sequence of instructions, \ 
re-write those instructions in the following format:

Step 1 - ...
Step 2 - …
…
Step N - …

If the text does not contain a sequence of instructions, \ 
then simply write \"No steps provided.\"

\"\"\"{text_2}\"\"\"
"""
response = get_completion(prompt)
print("Completion for Text 2:")
print(response)
```

### Original pre block 10

```text
Completion for Text 2:
No steps provided.
```

### Original pre block 11

```text
prompt = f"""
Your task is to answer in a consistent style.

<child>: Teach me about patience.

<grandparent>: The river that carves the deepest \ 
valley flows from a modest spring; the \ 
grandest symphony originates from a single note; \ 
the most intricate tapestry begins with a solitary thread.

<child>: Teach me about resilience.
"""
response = get_completion(prompt)
print(response)
```

### Original quote/transcript 1

```text
To guide a model towards the desired output and reduce irrelevant or incorrect responses, it is important to provide clear and specific instructions, which can be achieved through longer prompts that offer more clarity and context.
```

### Original quote/transcript 2

```text
<grandparent>: Resilience is like a mighty oak tree that withstands the strongest storms, bending but never breaking. It is the ability to bounce back from adversity, to find strength in the face of challenges, and to persevere even when the odds seem insurmountable. Just as a diamond is formed under immense pressure, resilience is forged through the trials and tribulations of life.
```